# 📄 PDF Parsing hàng loạt: MarkItDown + Qwen2.5-VL-7B (GPU)

**Mục tiêu:** tự động quét **toàn bộ file PDF** trong một thư mục dataset trên Kaggle,
và với mỗi file:

1. Trích xuất nhanh bằng `markitdown` (baseline).
2. Phát hiện trang "khó" (ít text / scan / cần OCR lại).
3. Dùng Qwen2.5-VL-7B để OCR lại các trang khó đó (giữ bảng markdown, công thức LaTeX,
   giữ đúng dấu tiếng Việt).
4. Ghép kết quả theo đúng thứ tự trang, xuất ra 1 file `.md` riêng cho mỗi PDF.

Model chỉ được load **một lần duy nhất** và tái sử dụng cho tất cả các file, tránh
lãng phí thời gian/VRAM.

**Yêu cầu:** Bật GPU trong Kaggle: `Settings > Accelerator > GPU`.


## Bước 0 — Kiểm tra GPU

In [ ]:
!nvidia-smi

## Bước 1 — Cài đặt thư viện

**Lưu ý quan trọng:** Kaggle tự động import sẵn nhiều thư viện (PIL, torchvision...)
ngay khi khởi động notebook, trước cả khi bạn chạy ô cài đặt bên dưới. Nếu ô này nâng
cấp các thư viện đó, Python vẫn giữ bản cũ trong bộ nhớ cho đến khi kernel được khởi
động lại, gây ra các lỗi khó hiểu kiểu `ImportError: cannot import name '_Ink'`.

→ **Sau khi chạy xong ô cài đặt, PHẢI restart kernel** (ô "Bước 1.1" ngay bên dưới sẽ
tự làm việc này) trước khi chạy tiếp các bước sau.


In [ ]:
!pip install -q 'markitdown[pdf]' pymupdf tqdm
!pip install -q "transformers>=4.49.0,<4.52.0" accelerate qwen-vl-utils
!pip install -q -U bitsandbytes
print("Cài đặt xong. Hãy chạy ô Bước 1.1 bên dưới để restart kernel.")


### Bước 1.1 — Restart kernel để áp dụng thay đổi

Chạy ô bên dưới, kernel sẽ tự tắt và Kaggle tự khởi động lại (mất khoảng 5-10 giây,
có thể thấy vòng xoay loading). **Sau khi kernel khởi động lại xong, chạy tiếp từ
Bước 2** — không cần chạy lại Bước 1 hay ô này lần nữa.


In [ ]:
import os
os.kill(os.getpid(), 9)

## Bước 2 — Quét toàn bộ file PDF trong thư mục dataset

> 💡 **Lưu ý quan trọng về lỗi tên file tiếng Việt trên Kaggle:**
> Khi bạn upload trực tiếp từng file PDF lên Kaggle Dataset qua trình duyệt, Kaggle sẽ **tự động lọc bỏ các nguyên âm có dấu tiếng Việt** trong tên file (ví dụ: `Mô hình chú ý ngữ cảnh đa tầm nhìn...` bị biến thành `M hnh ch  ng cnh a tm nhn...`).
> 
> **2 giải pháp đã được tích hợp sẵn trong notebook:**
> 1. **(Khuyên dùng khi đưa dữ liệu lên Kaggle)**: Nén các file PDF thành 1 file `.zip` (hoặc `.tar.gz`) trên máy tính trước, rồi upload file zip đó lên Kaggle. Ô bên dưới **tự động phát hiện và giải nén file zip**, bảo toàn 100% tên file tiếng Việt có dấu!
> 2. **Nếu đã upload trực tiếp khiến tên file bị mất dấu:** Notebook tích hợp thuật toán **tự động đọc trang 1 của PDF** bằng PyMuPDF để trích xuất lại chính xác Tiêu đề bài báo tiếng Việt (dựa trên font chữ lớn nhất). Tên thư mục kết quả, file `source_meta.json`, và file `article.md` đầu ra sẽ tự động mang tên chuẩn theo tiêu đề tiếng Việt thực sự của tài liệu.


In [ ]:
import os
import zipfile
import tarfile
import fitz  # PyMuPDF

# Thư mục chứa dataset trên Kaggle
# Mặc định quét /kaggle/input (tự động nhận diện mọi dataset được gắn vào notebook)
DATASET_DIR = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working/output"
EXTRACTED_DIR = "/kaggle/working/extracted_pdfs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(EXTRACTED_DIR, exist_ok=True)

# 1. Tự động kiểm tra và giải nén các file .zip/.tar nếu có trong dataset
# (Upload file .zip lên Kaggle là cách tốt nhất để bảo toàn 100% tên file tiếng Việt có dấu)
zip_extracted_count = 0
for root, dirs, files in os.walk(DATASET_DIR):
    for f in files:
        f_path = os.path.join(root, f)
        if f.lower().endswith(".zip"):
            try:
                with zipfile.ZipFile(f_path, 'r') as zf:
                    for member in zf.infolist():
                        try:
                            # Hỗ trợ giải nén file zip tiếng Việt từ Windows (CP437 -> UTF-8)
                            fname = member.filename.encode('cp437').decode('utf-8')
                        except Exception:
                            fname = member.filename
                        target_path = os.path.join(EXTRACTED_DIR, fname)
                        if member.is_dir():
                            os.makedirs(target_path, exist_ok=True)
                        else:
                            os.makedirs(os.path.dirname(target_path), exist_ok=True)
                            with zf.open(member) as sf, open(target_path, "wb") as df:
                                df.write(sf.read())
                    print(f"Đã giải nén zip: {f} -> {EXTRACTED_DIR}")
                    zip_extracted_count += 1
            except Exception as e:
                print(f"Lỗi khi giải nén {f}: {e}")
        elif f.lower().endswith((".tar.gz", ".tgz", ".tar")):
            try:
                with tarfile.open(f_path, 'r:*') as tf:
                    tf.extractall(EXTRACTED_DIR)
                    print(f"Đã giải nén tar: {f} -> {EXTRACTED_DIR}")
                    zip_extracted_count += 1
            except Exception as e:
                print(f"Lỗi khi giải nén {f}: {e}")

# 2. Quét tìm toàn bộ file PDF
pdf_files = []
scan_dirs = [DATASET_DIR]
if zip_extracted_count > 0:
    scan_dirs.append(EXTRACTED_DIR)

for scan_dir in scan_dirs:
    for root, dirs, files in os.walk(scan_dir):
        if os.path.abspath(root).startswith(os.path.abspath(OUTPUT_DIR)):
            continue
        for f in files:
            if f.lower().endswith(".pdf"):
                pdf_path = os.path.join(root, f)
                if pdf_path not in pdf_files:
                    pdf_files.append(pdf_path)

pdf_files = sorted(pdf_files)

def extract_pdf_title_fast(pdf_path):
    """Trích xuất nhanh tiêu đề tiếng Việt từ trang 1 của PDF bằng PyMuPDF."""
    try:
        doc = fitz.open(pdf_path)
        if len(doc) == 0:
            doc.close()
            return ""
        page = doc[0]
        blocks = page.get_text("dict").get("blocks", [])
        doc.close()
        spans_by_size = {}
        for b in blocks:
            if "lines" in b:
                for line in b["lines"]:
                    for span in line.get("spans", []):
                        text = span.get("text", "").strip()
                        size = round(span.get("size", 0), 1)
                        if not text or len(text) < 2 or text.isdigit():
                            continue
                        if text in [",", ".", "-", ":", ";", "/", "\\"]:
                            continue
                        spans_by_size.setdefault(size, []).append(text)
        if spans_by_size:
            import re
            for s in sorted(spans_by_size.keys(), reverse=True):
                cand = " ".join(spans_by_size[s]).strip()
                if len(cand) >= 15:
                    return re.sub(r"\s+", " ", cand)
    except Exception:
        pass
    return ""

print(f"\nTìm thấy {len(pdf_files)} file PDF:\n")
for i, p in enumerate(pdf_files):
    title = extract_pdf_title_fast(p)
    print(f"[{i}] {p}")
    if title:
        print(f"Tiêu đề tiếng Việt nhận diện: {title}")

assert len(pdf_files) > 0, f"Không tìm thấy file PDF nào trong {DATASET_DIR}!"


## Bước 3 — Load model Qwen2.5-VL-7B-Instruct

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

n_gpus = torch.cuda.device_count()
print(f"Số GPU khả dụng: {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}, {props.total_memory/1e9:.1f} GB")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

max_memory = {i: "11GiB" for i in range(max(n_gpus, 1))}
max_memory["cpu"] = "30GiB"

print("Đang tải model, có thể mất vài phút...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory,
    low_cpu_mem_usage=True,
)

MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 768 * 28 * 28
processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS,
)

processor.tokenizer.padding_side = "left"

print("Đã tải xong model.")
print("Phân bổ model trên các thiết bị:")
print(model.hf_device_map)

## Bước 4 — Các hàm dùng chung

- `render_pdf_pages`: render toàn bộ (hoặc 1 khoảng) trang PDF thành ảnh PNG.
- `render_pages_by_index`: render đúng những trang cần OCR (theo index), mở file PDF
  đúng 1 lần thay vì mở lại cho từng trang — tránh I/O thừa khi có nhiều trang cần VLM.
- `get_text_length_per_page` / `get_native_text_per_page`: lấy text gốc từ PyMuPDF.
- `ocr_pages_with_vlm`: gửi MỘT BATCH ảnh trang vào Qwen2.5-VL cùng lúc, nhận về danh
  sách Markdown tương ứng (tăng thông lượng GPU so với xử lý từng trang một).
- `ocr_page_with_vlm`: giữ lại để tương thích ngược (OCR 1 trang), bên trong gọi
  `ocr_pages_with_vlm` với batch = 1.


In [ ]:
import fitz  # PyMuPDF
from PIL import Image
import io
from qwen_vl_utils import process_vision_info

OCR_PROMPT = r"""
Bạn là hệ thống trích xuất tài liệu học thuật. Nhiệm vụ của bạn là đọc ảnh một trang PDF và chuyển đổi TẤT CẢ nội dung thành một mảng JSON (JSON array) chứa các block. KHÔNG xuất Markdown. CHỈ xuất JSON.

Mỗi block phải là một object có định dạng sau:
{
  "type": "heading" | "paragraph" | "list" | "formula" | "table" | "image" | "caption" | "metadata" | "reference" | "header_footer",
  "confidence": 0.95, // Hãy tự đánh giá độ tự tin (0.0 đến 1.0)
  // Các field khác tuỳ thuộc vào type:
}

1. type = "heading"
{
  "type": "heading",
  "level": 1, // 1 là to nhất, 2, 3...
  "text": "Nội dung heading",
  "confidence": 0.98
}

2. type = "paragraph"
{
  "type": "paragraph",
  "text": "Nội dung đoạn văn",
  "confidence": 0.95
}

3. type = "list"
{
  "type": "list",
  "items": ["mục 1", "mục 2"], // mảng các chuỗi
  "confidence": 0.95
}

4. type = "formula"
{
  "type": "formula",
  "latex": "công thức latex",
  "equation_number": "1" // nếu có số đánh dấu phương trình, nếu không có thì null,
  "confidence": 0.95
}

5. type = "table"
{
  "type": "table",
  "headers": ["Cột 1", "Cột 2"], // mảng string
  "rows": [
     ["Hàng 1 cột 1", "Hàng 1 cột 2"] // mảng các mảng string
  ],
  "confidence": 0.90
}

6. type = "image"
{
  "type": "image",
  "figure_id": "fig-1", // tự tạo một id duy nhất cho trang này, VD: fig-1
  "confidence": 0.95
}

7. type = "caption"
{
  "type": "caption",
  "text": "Hình 1. Minh họa mô hình",
  "confidence": 0.95
}

8. type = "metadata"
{
  "type": "metadata",
  "text": "Tác giả: Nguyễn Văn A...",
  "confidence": 0.95
}

9. type = "reference"
{
  "type": "reference",
  "text": "Tài liệu tham khảo 1...",
  "confidence": 0.95
}

10. type = "header_footer"
{
  "type": "header_footer",
  "text": "Tạp chí Khoa học... / Số trang: 247",
  "confidence": 0.99
}

Yêu cầu bắt buộc:
- TRÍCH XUẤT TOÀN BỘ TEXT, không bỏ sót.
- Giữ nguyên thứ tự đọc (từ trên xuống dưới, trái qua phải, theo cột nếu có).
- GIỮ NGUYÊN các dấu tiếng Việt, ký hiệu gốc.
- Đối với bảng, cố gắng giữ nguyên cấu trúc cột/hàng.
- Với công thức, viết bằng LaTeX chuẩn.
- Bắt buộc trả về định dạng mảng JSON hợp lệ, bắt đầu bằng `[` và kết thúc bằng `]`. KHÔNG bọc trong markdown code block (như ```json). CHỈ in ra JSON.
"""


def render_pdf_pages(pdf_path, dpi=300, page_range=None):
    doc = fitz.open(pdf_path)
    try:
        n_pages = len(doc)
        start, end = (0, n_pages) if page_range is None else page_range
        images = []
        for i in range(start, min(end, n_pages)):
            page = doc[i]
            pix = page.get_pixmap(dpi=dpi)
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            images.append(img)
        return images
    finally:
        doc.close()


def render_pages_by_index(pdf_path, dpi, indices):
    """Render đúng các trang cần thiết (theo index 0-based), mở file PDF một lần duy
    nhất thay vì mở lại cho từng trang. Trả về dict {index: PIL.Image}."""
    doc = fitz.open(pdf_path)
    try:
        images = {}
        for i in indices:
            page = doc[i]
            pix = page.get_pixmap(dpi=dpi)
            images[i] = Image.open(io.BytesIO(pix.tobytes("png")))
        return images
    finally:
        doc.close()


def get_text_length_per_page(pdf_path):
    doc = fitz.open(pdf_path)
    try:
        return [len(page.get_text().strip()) for page in doc]
    finally:
        doc.close()


def get_native_text_per_page(pdf_path):
    doc = fitz.open(pdf_path)
    try:
        return [page.get_text() for page in doc]
    finally:
        doc.close()


def ocr_pages_with_vlm(images, prompt=OCR_PROMPT, max_new_tokens=2048):
    """OCR một BATCH ảnh trang cùng lúc (thay vì gọi generate() riêng cho từng trang).
    Trả về list Markdown text, đúng thứ tự với `images`.

    Batch nhiều trang trong 1 lần forward pass giúp tận dụng GPU tốt hơn nhiều so với
    gọi generate() tuần tự cho từng trang một, đặc biệt khi model được chia qua nhiều
    GPU (device_map="auto") vì mỗi lần gọi tuần tự đều phải chờ toàn bộ pipeline chạy
    xong cho đúng 1 ảnh, gây "bubble" rảnh rỗi giữa các GPU.
    """
    messages_batch = [
        [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        for image in images
    ]
    texts = [
        processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in messages_batch
    ]
    image_inputs, video_inputs = process_vision_info(messages_batch)
    inputs = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    # inference_mode thay cho no_grad: tắt luôn cả version-counter tracking của
    # autograd (không chỉ tắt tính gradient), nhanh hơn no_grad một chút cho suy luận
    # thuần túy (không có chỗ nào cần backward ở đây).
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_texts = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_texts


def ocr_page_with_vlm(image, prompt=OCR_PROMPT, max_new_tokens=2048):
    """Giữ lại để tương thích ngược (API cũ, OCR 1 trang). Bên trong gọi hàm batch
    với batch = 1, nên hành vi/model call giống hệt bản gốc khi dùng đơn lẻ."""
    return ocr_pages_with_vlm([image], prompt=prompt, max_new_tokens=max_new_tokens)[0]

## Bước 5 — Cấu hình xử lý & phát hiện "trang khó" thông minh hơn

**Vấn đề với cách phát hiện chỉ dựa trên độ dài text:** các trang chứa công thức toán
bị lỗi (do PDF nhúng công thức bằng font Unicode toán học đặc biệt, khiến PyMuPDF đọc
ra chuỗi ký tự sai như `𝑉𝑉𝑐𝑐𝑐𝑐𝑐𝑐...`) vẫn có RẤT NHIỀU ký tự — chỉ là ký tự sai. Tương tự,
trang chứa bảng dài cũng có nhiều text. Nếu chỉ lọc theo "ít ký tự = trang khó", các
trang này sẽ bị bỏ sót, y hệt vấn đề bạn gặp phải.

→ Bổ sung các tiêu chí phát hiện dựa trên **nội dung**, không chỉ độ dài:
1. **Dò ký tự thuộc khối Unicode "Mathematical Alphanumeric Symbols"** (U+1D400–U+1D7FF)
   — đây chính xác là nguồn gốc gây lỗi công thức toán bạn thấy. Trang có nhiều ký tự
   trong dải này gần như chắc chắn chứa công thức bị lỗi.
2. **Heuristic nhận diện bảng**: trang có từ khóa "Bảng"/"Table" kèm mật độ số cao
   được coi là có khả năng chứa bảng cần OCR lại để giữ đúng cấu trúc.
3. **Tỷ lệ ký tự thay thế lỗi encoding** (`U+FFFD`, dấu hiệu PyMuPDF không giải mã được
   font nhúng trong PDF): trang có tỷ lệ ký tự này vượt ngưỡng cũng bị coi là cần VLM.

- `THRESHOLD`: trang có ít hơn N ký tự text gốc → coi là trang scan, cần VLM.
- `MATH_CHAR_THRESHOLD`: trang có từ N ký tự thuộc khối Unicode toán học trở lên → cần VLM.
- `REPLACEMENT_CHAR_THRESHOLD`: tỷ lệ ký tự lỗi encoding (U+FFFD) vượt ngưỡng → cần VLM.
- `FORCE_ALL_PAGES`: đặt `True` để ép xử lý TOÀN BỘ trang bằng VLM (đảm bảo chất lượng
  cao nhất, nhưng chậm hơn nhiều — dùng khi tài liệu quan trọng và không ngại đợi lâu).

Ngoài ra, mục này cũng khai báo 2 cấu hình tối ưu tốc độ mới:
- `VLM_BATCH_SIZE`: số trang OCR cùng lúc trong 1 lần gọi `model.generate()`. Nếu batch
  gây OOM, pipeline tự động chia đôi batch và thử lại (xem Bước 6), nên có thể đặt số
  này hơi lạc quan mà không sợ crash toàn bộ.
- `estimate_max_new_tokens(...)`: ước lượng ngân sách token sinh ra dựa trên độ dài
  native text của trang, thay vì luôn dùng `max_new_tokens=2048` cố định. Trang gần
  như không có native text (scan thật, không ước lượng được) vẫn giữ nguyên ngân sách
  đầy đủ để không cắt mất nội dung.


In [ ]:
import re
THRESHOLD = 50
MATH_CHAR_THRESHOLD = 15
REPLACEMENT_CHAR_THRESHOLD = 0.01  # tỷ lệ ký tự U+FFFD (lỗi encoding) -> cần VLM
FORCE_ALL_PAGES = False
MAX_PAGES_PER_FILE = None  # ví dụ đặt 30 nếu muốn giới hạn, None = không giới hạn
DPI = 300

# --- Cấu hình tối ưu tốc độ / VRAM cho bước VLM (Bước 6) ---
# GPU trong log gốc (T4 14.56GiB) gần hết VRAM ngay cả ở batch=1 (do ảnh trang không
# giới hạn pixel) -> bắt đầu an toàn ở batch=1. Sau khi đã áp dụng MIN_PIXELS/MAX_PIXELS
# ở Bước 3, có thể thử tăng lên 2 và theo dõi xem GPU còn dư VRAM hay không.
VLM_BATCH_SIZE = 1  # số trang OCR cùng lúc trong 1 lần generate(); tự fallback nếu OOM

MAX_NEW_TOKENS_DEFAULT = 4096  # tăng lên 4096 vì JSON output dài hơn Markdown
MAX_NEW_TOKENS_MIN = 768       # sàn tối thiểu, đủ cho 1 trang text bình thường
TOKENS_PER_NATIVE_CHAR = 0.9   # hệ số ước lượng hào phóng (token sinh ra / ký tự gốc)
ESTIMATE_MIN_NATIVE_LEN = 200  # dưới ngưỡng này coi là scan thật -> không ước lượng được,
                                # giữ nguyên MAX_NEW_TOKENS_DEFAULT để tránh cắt nội dung

# Khối Unicode "Mathematical Alphanumeric Symbols" - nguồn gốc phổ biến nhất gây lỗi
# công thức toán khi trích xuất PDF (Word Equation Editor / MathType thường xuất ra
# các kí tự trong dải này).
MATH_UNICODE_RANGES = [
    (0x1D400, 0x1D7FF),  # Mathematical Alphanumeric Symbols
    (0x2200, 0x22FF),    # Mathematical Operators
    (0x27C0, 0x27EF),    # Miscellaneous Mathematical Symbols-A
    (0x2980, 0x29FF),    # Miscellaneous Mathematical Symbols-B
]


def count_math_chars(text):
    count = 0
    for ch in text:
        cp = ord(ch)
        for lo, hi in MATH_UNICODE_RANGES:
            if lo <= cp <= hi:
                count += 1
                break
    return count


def looks_like_table(text):
    has_keyword = ("Bảng" in text) or ("Table" in text)
    tokens = text.split()
    if not tokens:
        return False
    numeric_tokens = sum(1 for t in tokens if re.search(r"\d", t))
    numeric_ratio = numeric_tokens / len(tokens)
    return has_keyword and numeric_ratio > 0.15


def replacement_char_ratio(text):
    """Tỷ lệ ký tự thay thế U+FFFD trong text — dấu hiệu PyMuPDF không giải mã được
    đúng font/encoding nhúng trong PDF (khác với lỗi font toán học, vốn ra ký tự
    Unicode hợp lệ nhưng sai nghĩa)."""
    if not text:
        return 0.0
    return text.count("\ufffd") / len(text)


def estimate_max_new_tokens(native_length):
    """Ước lượng ngân sách token sinh ra dựa trên độ dài native text của trang.
    Chỉ thu hẹp ngân sách khi trang đã có tương đối nhiều native text (dấu hiệu trang
    không phải scan trắng, chỉ bị lỗi font/bảng) — với trang gần như không có native
    text (scan thật, native_length rất nhỏ, không ước lượng được nội dung thật sự dài
    bao nhiêu) vẫn giữ nguyên ngân sách đầy đủ để không có rủi ro cắt mất nội dung."""
    if native_length < ESTIMATE_MIN_NATIVE_LEN:
        return MAX_NEW_TOKENS_DEFAULT
    estimated = int(native_length * TOKENS_PER_NATIVE_CHAR)
    return max(MAX_NEW_TOKENS_MIN, min(MAX_NEW_TOKENS_DEFAULT, estimated))


def classify_pages(pdf_path):
    """Trả về list dict mô tả từng trang: có cần VLM xử lý lại hay không, và vì sao."""
    doc = fitz.open(pdf_path)
    pages_info = []
    try:
        for i, page in enumerate(doc):
            text = page.get_text()
            length = len(text.strip())
            math_chars = count_math_chars(text)
            is_table = looks_like_table(text)
            rep_ratio = replacement_char_ratio(text)

            reasons = []
            if FORCE_ALL_PAGES:
                reasons.append("ép xử lý toàn bộ trang")
            if length < THRESHOLD:
                reasons.append("quá ít text, có thể là trang scan")
            if math_chars >= MATH_CHAR_THRESHOLD:
                reasons.append(f"phát hiện {math_chars} ký tự Unicode toán học")
            if is_table:
                reasons.append("phát hiện có thể có bảng")
            if rep_ratio >= REPLACEMENT_CHAR_THRESHOLD:
                reasons.append(f"phát hiện lỗi encoding (tỷ lệ U+FFFD: {rep_ratio:.3f})")

            pages_info.append({
                "index": i,
                "length": length,
                "math_chars": math_chars,
                "looks_like_table": is_table,
                "replacement_ratio": rep_ratio,
                "needs_vlm": len(reasons) > 0,
                "reasons": reasons,
            })
        return pages_info
    finally:


## Bước 6 — Hàm xử lý 1 file PDF hoàn chỉnh

Hàm này gộp toàn bộ pipeline (markitdown baseline → phát hiện trang khó → OCR bằng
VLM theo batch → ghép kết quả → lưu file) cho **một** file PDF. Bước 7 sẽ gọi hàm này
lặp lại cho tất cả file trong `pdf_files`.

So với bản trước, phần OCR bằng VLM giờ:
- Render đúng những trang cần OCR trong 1 lần mở file (`render_pages_by_index`), thay
  vì mở lại file PDF cho từng trang.
- Gộp `VLM_BATCH_SIZE` trang vào 1 lần gọi `model.generate()` (`ocr_pages_with_vlm`)
  thay vì gọi tuần tự từng trang.
- Tự động chia đôi batch và thử lại nếu gặp OOM, cho tới khi về batch = 1; nếu batch =
  1 vẫn OOM thì trang đó được ghi nhận lỗi (không crash toàn bộ file).
- Ước lượng `max_new_tokens` theo độ dài native text thay vì luôn dùng 2048.
- Đo thời gian từng giai đoạn (native extraction, quality check, render, VLM
  inference) để phục vụ benchmark ở Bước 7/8. Hàm trả về thêm `bench` (dict) bên cạnh
  `final_path` và `log` như trước.


In [ ]:
import json
import time
import shutil
import hashlib
import re
import unicodedata
import fitz
from markitdown import MarkItDown

md_converter = MarkItDown()


def safe_slug(text, max_len=60):
    """Chuyển chuỗi tiếng Việt thành slug ASCII chuẩn, an toàn cho tên thư mục / file."""
    text = text.replace("Đ", "D").replace("đ", "d")
    normalized = unicodedata.normalize("NFKD", text)
    ascii_text = normalized.encode("ascii", "ignore").decode("ascii")
    ascii_text = re.sub(r"[^A-Za-z0-9]+", "_", ascii_text)
    ascii_text = re.sub(r"_+", "_", ascii_text).strip("_")
    return ascii_text[:max_len].rstrip("_") or "document"


def safe_filename(path, max_len=90):
    """Return an ASCII-only, collision-resistant slug for output folders."""
    base = os.path.splitext(os.path.basename(path))[0]
    ascii_base = safe_slug(base, max_len=60)
    reserved = {"CON", "PRN", "AUX", "NUL", *(f"COM{i}" for i in range(1, 10)), *(f"LPT{i}" for i in range(1, 10))}
    if ascii_base.upper() in reserved:
        ascii_base = f"{ascii_base}_file"

    path_hash = hashlib.sha1(os.path.abspath(path).encode("utf-8", "surrogatepass")).hexdigest()[:8]
    keep = max(1, max_len - len(path_hash) - 1)
    return f"{ascii_base[:keep].rstrip('._-')}_{path_hash}"


def file_hash(path):
    return hashlib.sha1(os.path.abspath(path).encode("utf-8", "surrogatepass")).hexdigest()[:8]


# Khôi phục thủ công tên bài báo tiếng Việt (nếu muốn chỉ định cứng)
# Hỗ trợ cả file_hash hoặc chuỗi con trong tên file (không phân biệt    hoa thường)
PDF_NAME_OVERRIDES = {
    # Ví dụ:
    # "5e751ba5": "Mô hình chú ý ngữ cảnh đa tầm nhìn cải tiến cho bài toán trả lời câu hỏi dựa trên hình ảnh bằng tiếng Việt",
    # "m hnh ch  ng cnh": "Mô hình chú ý ngữ cảnh đa tầm nhìn cải tiến cho bài toán trả lời câu hỏi dựa trên hình ảnh bằng tiếng Việt",
}


def extract_pdf_title(pdf_path):
    """
    Tự động trích xuất tiêu đề tiếng Việt chuẩn từ trang 1 của PDF bằng PyMuPDF.
    Dò tìm cụm từ có font-size lớn nhất (tiêu đề bài báo học thuật).
    """
    # 1. Kiểm tra overrides
    key = file_hash(pdf_path)
    if key in PDF_NAME_OVERRIDES:
        return PDF_NAME_OVERRIDES[key]
    
    basename = os.path.basename(pdf_path)
    for k, v in PDF_NAME_OVERRIDES.items():
        if k.lower() in basename.lower():
            return v

    # 2. Quét trang 1 tìm font lớn nhất
    try:
        doc = fitz.open(pdf_path)
        if len(doc) > 0:
            page = doc[0]
            blocks = page.get_text("dict").get("blocks", [])
            spans_by_size = {}
            for b in blocks:
                if "lines" in b:
                    for line in b["lines"]:
                        for span in line.get("spans", []):
                            text = span.get("text", "").strip()
                            size = round(span.get("size", 0), 1)
                            if not text or len(text) < 2 or text.isdigit():
                                continue
                            if text in [",", ".", "-", ":", ";", "/", "\\"]:
                                continue
                            spans_by_size.setdefault(size, []).append(text)
            doc.close()
            
            if spans_by_size:
                for size in sorted(spans_by_size.keys(), reverse=True):
                    candidate = " ".join(spans_by_size[size]).strip()
                    if len(candidate) >= 15:
                        candidate = re.sub(r"\s+", " ", candidate)
                        if not any(kw in candidate.lower() for kw in ["email:", "tác giả liên hệ", "ngày nhận bài"]):
                            return candidate
    except Exception:
        pass

    # 3. Kiểm tra metadata PDF nếu hợp lệ
    try:
        doc = fitz.open(pdf_path)
        title = (doc.metadata or {}).get("title", "").strip()
        doc.close()
        junk_titles = {
            "untitled", "unknown", "jmst_template",
            "on the existence of solutions for vector quasiequilibrium problems"
        }
        if title and title.lower() not in junk_titles and len(title) > 3:
            return title
    except Exception:
        pass

    # 4. Fallback: dùng tên file gốc
    return os.path.splitext(basename)[0]


def get_pdf_display_name(pdf_path):
    """Trả về tiêu đề bài báo tiếng Việt chuẩn xác (ưu tiên tiêu đề trích xuất từ trang 1)."""
    title = extract_pdf_title(pdf_path)
    basename = os.path.basename(pdf_path)
    # Nhận diện nếu tên file bị Kaggle làm hỏng (chứa 2 dấu cách liên tiếp hoặc mất hết nguyên âm)
    is_corrupted = ("  " in basename) or (len(re.findall(r"\b[b-df-hj-np-tv-z]{2,}\b", basename.lower())) >= 2)
    if title and (is_corrupted or title.lower() not in basename.lower()):
        return title
    return os.path.splitext(basename)[0]


def output_dir_for_pdf(idx, pdf_path):
    """
    Tạo thư mục kết quả an toàn và dễ đọc: STT + slug tiếng Việt không dấu + hash.
    Ví dụ: 00_Mo_hinh_chu_y_ngu_canh_da_tam_nhin_cai_tien_5e751ba5
    """
    title = get_pdf_display_name(pdf_path)
    slug = safe_slug(title, max_len=45)
    p_hash = file_hash(pdf_path)
    return os.path.join(OUTPUT_DIR, f"{idx:02d}_{slug}_{p_hash}")


def process_single_pdf(pdf_path, file_output_dir, verbose=True, pbar=None, file_prefix=""):
    os.makedirs(file_output_dir, exist_ok=True)
    log = []
    bench = {
        "total_pages": 0,
        "native_pages": 0,
        "vlm_pages": 0,
        "t_native_extract": 0.0,
        "t_quality_check": 0.0,
        "t_render": 0.0,
        "t_vlm": 0.0,
    }

    def log_event(message):
        log.append(message)
        if verbose:
            try:
                from tqdm.auto import tqdm
                tqdm.write(f"  {message}")
            except Exception:
                print(" ", message, flush=True)

    original_pdf_path = pdf_path
    safe_pdf_path = os.path.join(file_output_dir, "source.pdf")
    source_meta_path = os.path.join(file_output_dir, "source_meta.json")
    doc_title = get_pdf_display_name(original_pdf_path)
    source_meta = {
        "title": doc_title,
        "display_name": doc_title,
        "source_hash": file_hash(original_pdf_path),
        "original_path": original_pdf_path,
        "original_basename": os.path.basename(original_pdf_path),
        "original_size": os.path.getsize(original_pdf_path),
        "original_mtime": os.path.getmtime(original_pdf_path),
    }
    need_copy = True
    if os.path.exists(safe_pdf_path) and os.path.exists(source_meta_path):
        try:
            with open(source_meta_path, "r", encoding="utf-8") as f:
                old_meta = json.load(f)
            need_copy = old_meta != source_meta
        except Exception:
            need_copy = True
    if need_copy:
        shutil.copyfile(original_pdf_path, safe_pdf_path)
        with open(source_meta_path, "w", encoding="utf-8") as f:
            json.dump(source_meta, f, ensure_ascii=False, indent=2)
        log_event(f"Tài liệu: {doc_title}")
        log_event(f"Đã tạo bản copy an toàn: {safe_pdf_path}")
    else:
        log_event(f"Tài liệu: {doc_title}")
        log_event(f"Dùng lại bản copy an toàn: {safe_pdf_path}")
    pdf_path = safe_pdf_path  # from here on, use the ASCII-safe copy

    # --- 1. Baseline bằng markitdown ---
    try:
        result = md_converter.convert(pdf_path)
        baseline_markdown = result.text_content
    except Exception as e:
        baseline_markdown = ""
        log_event(f"[LỖI markitdown] {e}")

    with open(os.path.join(file_output_dir, "baseline_markitdown.md"), "w", encoding="utf-8") as f:
        f.write(baseline_markdown)

    if pbar:
        pbar.update(1)  # Hoàn tất Bước 1: MarkItDown baseline (1/5)
        if file_prefix:
            pbar.set_description(f"{file_prefix}: PDF processing")

    # --- 2. Phát hiện trang khó (dựa trên độ dài + công thức toán + bảng + encoding lỗi) ---
    doc = fitz.open(pdf_path)
    try:
        total_pages = len(doc)
    finally:
        doc.close()

    t0 = time.time()
    pages_info = classify_pages(pdf_path)
    bench["t_quality_check"] = time.time() - t0

    t0 = time.time()
    native_texts = get_native_text_per_page(pdf_path)
    bench["t_native_extract"] = time.time() - t0

    pages_needing_vlm = [p["index"] for p in pages_info if p["needs_vlm"]]
    if MAX_PAGES_PER_FILE is not None:
        pages_needing_vlm = pages_needing_vlm[:MAX_PAGES_PER_FILE]

    bench["total_pages"] = total_pages
    bench["vlm_pages"] = len(pages_needing_vlm)
    bench["native_pages"] = total_pages - len(pages_needing_vlm)

    log_event(f"Tổng số trang: {total_pages}")
    log_event(f"Số trang cần VLM xử lý: {len(pages_needing_vlm)} -> {pages_needing_vlm}")
    for p in pages_info:
        if p["needs_vlm"]:
            log_event(f"  Trang {p['index']+1}: lý do = {', '.join(p['reasons'])}")

    # --- 3. OCR bằng VLM theo batch cho các trang khó, lưu tiến độ liên tục ---
    progress_file = os.path.join(file_output_dir, "vlm_progress.json")
    vlm_results = {}
    if os.path.exists(progress_file):
        try:
            with open(progress_file, "r", encoding="utf-8") as f:
                vlm_results = {int(k): v for k, v in json.load(f).items()}
            log_event(f"Đã nạp progress cũ: {len(vlm_results)} trang")
        except Exception as e:
            backup_file = progress_file + f".broken_{int(time.time())}"
            shutil.move(progress_file, backup_file)
            log_event(f"Progress JSON bị lỗi, đã backup sang {backup_file}: {e}")

    def save_progress():
        tmp_progress_file = progress_file + ".tmp"
        with open(tmp_progress_file, "w", encoding="utf-8") as f:
            json.dump(vlm_results, f, ensure_ascii=False, indent=2)
        os.replace(tmp_progress_file, progress_file)

    def run_batch(batch_indices):
        if not batch_indices:
            return

        t_r0 = time.time()
        images_by_idx = render_pages_by_index(pdf_path, DPI, batch_indices)
        bench["t_render"] += time.time() - t_r0

        budget = max(estimate_max_new_tokens(len(native_texts[i])) for i in batch_indices)
        imgs = [images_by_idx[i] for i in batch_indices]

        t_v0 = time.time()
        try:
            outputs = ocr_pages_with_vlm(imgs, max_new_tokens=budget)
            bench["t_vlm"] += time.time() - t_v0
            for idx, text_out in zip(batch_indices, outputs):
                vlm_results[idx] = text_out
            log_event(
                f"  Batch trang {[i+1 for i in batch_indices]}: xong "
                f"({time.time()-t_v0:.1f}s, batch_size={len(batch_indices)}, budget={budget})"
            )
            save_progress()
        except RuntimeError as e:
            bench["t_vlm"] += time.time() - t_v0
            if "out of memory" not in str(e).lower():
                raise
            torch.cuda.empty_cache()
            if len(batch_indices) == 1:
                idx = batch_indices[0]
                vlm_results[idx] = f"[LỖI OCR trang {idx+1}: OOM ngay cả với batch=1: {e}]"
                log_event(f"  Trang {idx+1}: OOM ngay cả batch=1, bỏ qua trang này - {e}")
                save_progress()
            else:
                mid = len(batch_indices) // 2
                log_event(
                    f"  Batch trang {[i+1 for i in batch_indices]}: OOM (batch_size="
                    f"{len(batch_indices)}) -> tự chia đôi và thử lại"
                )
                run_batch(batch_indices[:mid])
                run_batch(batch_indices[mid:])
        except Exception as e:
            bench["t_vlm"] += time.time() - t_v0
            if len(batch_indices) == 1:
                idx = batch_indices[0]
                vlm_results[idx] = f"[LỖI OCR trang {idx+1}: {e}]"
                log_event(f"  Trang {idx+1}: LỖI - {e}")
                save_progress()
            else:
                log_event(
                    f"  Batch trang {[i+1 for i in batch_indices]}: LỖI ({e}) -> "
                    f"xử lý lại từng trang một để cô lập lỗi"
                )
                for idx in batch_indices:
                    run_batch([idx])

    pending = [i for i in pages_needing_vlm if i not in vlm_results]
    for start in range(0, len(pending), VLM_BATCH_SIZE):
        run_batch(pending[start:start + VLM_BATCH_SIZE])

    # --- 4. Ghép kết quả cuối cùng thành mảng JSON ---
    final_blocks = []
    for i in range(total_pages):
        page_content = vlm_results.get(i, native_texts[i])
        page_content = re.sub(r'^```json\s*', '', page_content)
        page_content = re.sub(r'\s*```$', '', page_content)
        page_content = page_content.strip()
        
        try:
            blocks = json.loads(page_content)
            if isinstance(blocks, list):
                for block in blocks:
                    block['page'] = i + 1
                    block['source'] = 'explicit'
                    final_blocks.append(block)
            else:
                final_blocks.append({"type": "paragraph", "text": str(blocks), "page": i + 1, "source": "fallback", "confidence": 0.5})
        except Exception as e:
            final_blocks.append({"type": "paragraph", "text": page_content, "page": i + 1, "source": "fallback", "confidence": 0.0})

    final_path = os.path.join(file_output_dir, "article_raw.json")
    with open(final_path, "w", encoding="utf-8") as f:
        json.dump(final_blocks, f, ensure_ascii=False, indent=2)

    log_event(f"Đã lưu: {final_path}")

    with open(os.path.join(file_output_dir, "log.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(log))

    return final_path, log, bench


## Bước 6.1 — Các hàm Post-processing (Phase 3, 4, 5)

In [ ]:
import json
import os
import re
import fitz

def reconstruct_document(raw_json_path: str, output_dir: str) -> str:
    """
    Phase 3: nhận article_raw.json, trả về article.json đã làm sạch + ghép.
    """
    with open(raw_json_path, "r", encoding="utf-8") as f:
        blocks = json.load(f)

    # 1. Lọc header/footer
    blocks = [b for b in blocks if b.get("type") != "header_footer"]

    # 2. Ghép paragraph bị ngắt giữa 2 trang liền nhau.
    #    Dấu hiệu: paragraph cuối trang không kết thúc bằng dấu câu, và
    #    paragraph đầu trang sau bắt đầu bằng chữ thường.
    merged = []
    i = 0
    while i < len(blocks):
        blk = dict(blocks[i])
        if (
            blk.get("type") == "paragraph"
            and i + 1 < len(blocks)
            and blocks[i + 1].get("type") == "paragraph"
        ):
            cur_text = blk.get("text", "")
            next_text = blocks[i + 1].get("text", "")
            # Ghép nếu dòng cuối không kết thúc bằng dấu câu
            # và dòng tiếp theo trên trang khác bắt đầu bằng chữ thường hoặc số
            ends_mid = bool(cur_text) and cur_text[-1] not in ".!?:;)»\u201d\u2019"
            starts_lower = bool(next_text) and (next_text[0].islower() or next_text[0].isdigit())
            cross_page = blk.get("page", 0) != blocks[i + 1].get("page", 0)
            if ends_mid and starts_lower and cross_page:
                blk["text"] = cur_text.rstrip() + " " + next_text.lstrip()
                blk["page_end"] = blocks[i + 1].get("page", blk.get("page"))
                blk["merged"] = True
                blk["source"] = "inferred"
                i += 2
                merged.append(blk)
                continue
        merged.append(blk)
        i += 1

    # 3. Xác định heading hierarchy từ level đã có,
    #    nếu thiếu level thì infer từ vị trí (trang đầu = h1)
    for blk in merged:
        if blk.get("type") == "heading" and "level" not in blk:
            blk["level"] = 2
            blk["source"] = "inferred"

    # 4. Gắn sequence index để giữ thứ tự
    for idx, blk in enumerate(merged):
        blk["seq"] = idx

    # 5. Xuất article.json
    out_path = os.path.join(output_dir, "article.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(merged, f, ensure_ascii=False, indent=2)

    try:
        from tqdm.auto import tqdm
        tqdm.write(f"  [Phase 3] Saved {len(merged)} blocks → {out_path}")
    except Exception:
        print(f"  [Phase 3] Saved {len(merged)} blocks → {out_path}")
    return out_path

def extract_figures(pdf_path: str, article_json_path: str, output_dir: str) -> dict:
    """
    Với mỗi block type='image', tìm ảnh nhúng trong PDF trang tương ứng và lưu.
    Trả về dict figure_id → saved_path (relative to output_dir).
    """
    with open(article_json_path, "r", encoding="utf-8") as f:
        blocks = json.load(f)

    image_blocks = [b for b in blocks if b.get("type") == "image"]
    if not image_blocks:
        return {}

    figures_dir = os.path.join(output_dir, "figures")
    os.makedirs(figures_dir, exist_ok=True)

    doc = fitz.open(pdf_path)
    saved = {}
    try:
        for blk in image_blocks:
            fig_id = blk.get("figure_id", f"fig-{blk.get('seq', 0)}")
            page_num = blk.get("page", 1) - 1  # 0-based
            page = doc[page_num] if page_num < len(doc) else doc[0]
            img_list = page.get_images(full=True)
            if not img_list:
                saved[fig_id] = None
                continue
            # Lấy ảnh lớn nhất trên trang (heuristic)
            best = max(img_list, key=lambda x: x[2] * x[3])  # width * height
            xref = best[0]
            base_img = doc.extract_image(xref)
            ext = base_img.get("ext", "png")
            fname = f"{fig_id}.{ext}"
            fpath = os.path.join(figures_dir, fname)
            with open(fpath, "wb") as fw:
                fw.write(base_img["image"])
            saved[fig_id] = f"figures/{fname}"
            try:
                from tqdm.auto import tqdm
                tqdm.write(f"  [Phase 4] Extracted {fig_id} → {fpath}")
            except Exception:
                print(f"  [Phase 4] Extracted {fig_id} → {fpath}")
    finally:
        doc.close()
    return saved

def render_markdown(article_json_path: str, figure_map: dict) -> str:
    """
    Python renderer: JSON blocks → Markdown string.
    """
    with open(article_json_path, "r", encoding="utf-8") as f:
        blocks = json.load(f)

    lines = []
    for blk in blocks:
        btype = blk.get("type")

        if btype == "heading":
            level = blk.get("level", 2)
            prefix = "#" * max(1, min(level, 4))
            lines.append(f"{prefix} {blk.get('text', '').strip()}")
            lines.append("")

        elif btype == "paragraph":
            text = blk.get("text", "").strip()
            if text:
                lines.append(text)
                lines.append("")

        elif btype == "list":
            items = blk.get("items", [])
            for item in items:
                lines.append(f"- {item.strip()}")
            lines.append("")

        elif btype == "formula":
            latex = blk.get("latex", "").strip()
            eq_num = blk.get("equation_number")
            if eq_num:
                lines.append(f"$$\n{latex} \\tag{{{eq_num}}}\n$$")
            else:
                lines.append(f"$$\n{latex}\n$$")
            lines.append("")

        elif btype == "table":
            headers = blk.get("headers", [])
            rows = blk.get("rows", [])
            if headers:
                lines.append("| " + " | ".join(str(h) for h in headers) + " |")
                lines.append("| " + " | ".join("---" for _ in headers) + " |")
            for row in rows:
                lines.append("| " + " | ".join(str(c) for c in row) + " |")
            lines.append("")

        elif btype == "image":
            fig_id = blk.get("figure_id", "fig")
            img_path = figure_map.get(fig_id)
            # Caption sẽ được thêm bởi block caption tiếp theo
            if img_path:
                lines.append(f"![]({img_path})")
            else:
                lines.append(f"![Hình: {fig_id}]")
            lines.append("")

        elif btype == "caption":
            text = blk.get("text", "").strip()
            if text:
                lines.append(f"*{text}*")
                lines.append("")

        elif btype == "metadata":
            text = blk.get("text", "").strip()
            if text:
                lines.append(text)
                lines.append("")

        elif btype == "reference":
            text = blk.get("text", "").strip()
            if text:
                lines.append(text)
                lines.append("")

        # header_footer đã bị lọc ở Phase 3, bỏ qua ở đây
    
    # Dọn dẹp nhiều dòng trống liên tiếp
    md = "\n".join(lines)
    md = re.sub(r"\n{3,}", "\n\n", md)
    return md.strip()

def validate_markdown(md_path: str) -> list:
    """Trả về list cảnh báo (warnings) nếu có vấn đề."""
    warnings = []
    with open(md_path, "r", encoding="utf-8") as f:
        content = f.read()

    # 1. Kiểm tra $$ cân bằng
    dd_count = content.count("$$")
    if dd_count % 2 != 0:
        warnings.append({"check": "formula_delimiter", "issue": f"Số cặp $$ lẻ: {dd_count}"})

    # 2. Kiểm tra không còn code fence thừa ```
    raw_fences = re.findall(r"```", content)
    if len(raw_fences) % 2 != 0:
        warnings.append({"check": "code_fence", "issue": f"Code fence ``` lẻ: {len(raw_fences)}"})

    # 3. Kiểm tra heading hierarchy không nhảy quá 1 cấp
    headings = re.findall(r"^(#{1,4})\s", content, re.MULTILINE)
    prev_level = 0
    for h in headings:
        level = len(h)
        if level > prev_level + 1 and prev_level > 0:
            warnings.append({"check": "heading_hierarchy", "issue": f"Nhảy từ H{prev_level} lên H{level}"})
        prev_level = level

    # 4. Kiểm tra header/footer lặp (chữ xuất hiện > 5 lần)
    footer_candidates = re.findall(r"^Tạp chí Khoa học|^Bùi Anh Đài", content, re.MULTILINE)
    if len(footer_candidates) > 3:
        warnings.append({"check": "repeated_header_footer", "issue": f"Header/footer có thể còn lặp: {len(footer_candidates)} lần"})

    return warnings

def build_repair_log(article_json_path: str) -> list:
    """Ghi lại tất cả block có confidence < 0.7 vào repair log."""
    with open(article_json_path, "r", encoding="utf-8") as f:
        blocks = json.load(f)
    
    low_conf = []
    for blk in blocks:
        conf = blk.get("confidence", 1.0)
        if conf < 0.7:
            low_conf.append({
                "seq": blk.get("seq"),
                "type": blk.get("type"),
                "page": blk.get("page"),
                "confidence": conf,
                "original": blk.get("text") or blk.get("latex") or str(blk),
                "repaired": None,  # Phase 5 repair chưa implement auto-fix
                "reason": "low_confidence_vlm_extraction"
            })
    return low_conf



## Bước 7 — Chạy hàng loạt cho tất cả file PDF

Mỗi file sẽ có 1 thư mục con riêng trong `OUTPUT_DIR`, chứa:
`baseline_markitdown.md`, `vlm_progress.json`, `article_raw.json`, `article.json`, `article.md`, `log.txt`, `repair_log.json`.

**Thanh tiến trình kép (tqdm):**
- **Tổng tiến trình:** Thể hiện % và số lượng file đã xử lý / tổng số file PDF trong dataset.
- **Tiến trình từng file (5 bước):**
  1. `MarkItDown baseline` (1/5)
  2. `PDF processing` (2/5): VLM OCR & trích xuất block
  3. `Reconstruction` (3/5): Nối đoạn văn bản, xác định cấu trúc heading
  4. `Image extraction` (4/5): Cắt ảnh nhúng & kết xuất Markdown
  5. `DONE` (5/5): Kiểm tra tính hợp lệ & tạo Repair Log

Nếu Kaggle session bị ngắt giữa chừng, chạy lại ô này sẽ tự động bỏ qua các trang đã xử lý xong nhờ `vlm_progress.json`.


In [ ]:
import json
import os
import time
from tqdm.auto import tqdm

summary = []

global_bench = {
    "total_pages": 0,
    "native_pages": 0,
    "vlm_pages": 0,
    "t_native_extract": 0.0,
    "t_quality_check": 0.0,
    "t_render": 0.0,
    "t_vlm": 0.0,
}
pipeline_t0 = time.time()
total_files = len(pdf_files)

# 1. Thanh tiến trình tổng cho toàn bộ dataset
overall_pbar = tqdm(total=total_files, desc="Tổng tiến trình", unit="file")

for idx, pdf_path in enumerate(pdf_files):
    file_output_dir = output_dir_for_pdf(idx, pdf_path)
    display_name = get_pdf_display_name(pdf_path)
    source_hash = file_hash(pdf_path)
    file_prefix = f"File {idx+1}/{total_files}"

    tqdm.write(f"\n{'='*60}")
    tqdm.write(f"[{idx+1}/{total_files}] Đang xử lý: {display_name}")
    tqdm.write(f"  • File gốc: {os.path.basename(pdf_path)}")
    tqdm.write(f"  • Source hash: {source_hash}")
    tqdm.write(f"  • Thư mục output: {file_output_dir}")
    tqdm.write(f"{'='*60}\n")

    # 2. Thanh tiến trình chi tiết cho từng file (5 giai đoạn: 1/5 -> 5/5)
    file_pbar = tqdm(total=5, desc=f"{file_prefix}: MarkItDown baseline", leave=False)

    try:
        # Giai đoạn 1 (1/5: MarkItDown) & Giai đoạn 2 (2/5: PDF processing)
        final_path, log, bench = process_single_pdf(
            pdf_path, file_output_dir, verbose=True, pbar=file_pbar, file_prefix=file_prefix
        )
        file_pbar.update(1)  # Hoàn tất Giai đoạn 2: PDF processing (2/5)

        # Giai đoạn 3 (3/5: Reconstruction)
        file_pbar.set_description(f"{file_prefix}: Reconstruction")
        article_json = reconstruct_document(final_path, file_output_dir)
        file_pbar.update(1)  # Hoàn tất Giai đoạn 3: Reconstruction (3/5)

        # Giai đoạn 4 (4/5: Image extraction & Render Markdown)
        file_pbar.set_description(f"{file_prefix}: Image extraction")
        safe_pdf = os.path.join(file_output_dir, "source.pdf")
        figure_map = extract_figures(safe_pdf, article_json, file_output_dir)
        md_content = render_markdown(article_json, figure_map)
        
        md_path = os.path.join(file_output_dir, "article.md")
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(md_content)
        tqdm.write(f"  [Post-process] Saved Markdown → {md_path}")
        file_pbar.update(1)  # Hoàn tất Giai đoạn 4: Image extraction (4/5)

        # Giai đoạn 5 (5/5: DONE - Validation & Repair Log)
        file_pbar.set_description(f"{file_prefix}: DONE")
        warnings = validate_markdown(md_path)
        repair_log = build_repair_log(article_json)
        if warnings:
            tqdm.write(f"  [Post-process] Validation warnings: {len(warnings)}")
        
        log_path = os.path.join(file_output_dir, "repair_log.json")
        with open(log_path, "w", encoding="utf-8") as f:
            json.dump({
                "validation_warnings": warnings,
                "low_confidence_blocks": repair_log
            }, f, ensure_ascii=False, indent=2)
            
        file_pbar.update(1)  # Hoàn tất Giai đoạn 5: DONE (5/5)
        file_pbar.close()
        
        summary.append({
            "index": idx,
            "title": display_name,
            "folder": os.path.basename(file_output_dir),
            "file": pdf_path,
            "status": "OK",
            "article_md": md_path,
            "source_hash": source_hash,
        })
        for k in global_bench:
            global_bench[k] += bench[k]
    except Exception as e:
        file_pbar.set_description(f"{file_prefix}: LỖI")
        file_pbar.close()
        tqdm.write(f"LỖI TOÀN BỘ FILE: {e}")
        summary.append({
            "index": idx,
            "title": display_name,
            "folder": os.path.basename(file_output_dir),
            "file": pdf_path,
            "status": "FAILED",
            "error": str(e),
            "source_hash": source_hash,
        })
    finally:
        overall_pbar.update(1)

overall_pbar.close()
pipeline_total = time.time() - pipeline_t0

print("\n\nHOÀN TẤT TOÀN BỘ.")

print("\nBENCHMARK\n")
print(f"Total pages: {global_bench['total_pages']}\n")
print(f"Native pages: {global_bench['native_pages']}")
print(f"VLM pages: {global_bench['vlm_pages']}\n")
print(f"Native extraction: {global_bench['t_native_extract']:.1f} sec")
print(f"Quality check: {global_bench['t_quality_check']:.1f} sec")
print(f"Rendering: {global_bench['t_render']:.1f} sec")
print(f"VLM inference: {global_bench['t_vlm']:.1f} sec\n")
print(f"Total: {pipeline_total:.1f} sec")
if global_bench["total_pages"] > 0:
    print(f"Average: {pipeline_total/global_bench['total_pages']:.2f} sec/page")
    print(f"VLM usage: {100*global_bench['vlm_pages']/global_bench['total_pages']:.1f}%")


## Bước 8 — Xem tổng kết

In [ ]:
import pandas as pd

df_summary = pd.DataFrame(summary)
if not df_summary.empty:
    cols = [c for c in ["index", "title", "folder", "status", "article_md"] if c in df_summary.columns]
    display(df_summary[cols])
else:
    print("Chưa có dữ liệu tổng kết.")


## Bước 9 — Preview kết quả cuối

Xem nhanh `article.md` và tóm tắt pipeline.


In [ ]:
preview_idx = 0

if summary and preview_idx < len(summary):
    output_dir = os.path.dirname(summary[preview_idx]["article_md"])
    md_path = os.path.join(output_dir, "article.md")
    json_path = os.path.join(output_dir, "article.json")
    repair_path = os.path.join(output_dir, "repair_log.json")

    print("=" * 60)
    print(f"TIÊU ĐỀ: {summary[preview_idx].get('title', '')}")
    print(f"OUTPUT DIR: {output_dir}")
    print("=" * 60)

    if os.path.exists(md_path):
        with open(md_path, "r", encoding="utf-8") as f:
            md = f.read()
        print(f"\n[article.md] {len(md)} chars\n")
        print(md[:3000])
        print("...\n")

    if os.path.exists(repair_path):
        with open(repair_path, "r", encoding="utf-8") as f:
            rlog = json.load(f)
        print(f"[repair_log.json]")
        print(f"  Validation warnings: {len(rlog.get('validation_warnings', []))}")
        print(f"  Low-confidence blocks: {len(rlog.get('low_confidence_blocks', []))}")
else:
    print("Chưa có file nào được xử lý.")


## Bước 10 — Nén toàn bộ kết quả để tải về

Vào tab **Output** bên phải notebook Kaggle để tải file zip này.


In [ ]:
import shutil
import json

# Tạo file README.md và summary.json trong thư mục output để khi giải nén tra cứu tiện lợi
readme_path = os.path.join(OUTPUT_DIR, "README.md")
with open(readme_path, "w", encoding="utf-8") as f:
    f.write("# DANH MỤC KẾT QUẢ PARSE PDF SANG MARKDOWN\n\n")
    f.write("| STT | Tiêu đề bài báo tiếng Việt | Thư mục kết quả | Trạng thái |\n")
    f.write("| :---: | :--- | :--- | :---: |\n")
    for s in summary:
        f.write(f"| {s.get('index', 0) + 1} | **{s.get('title', '')}** | `{s.get('folder', '')}` | {s.get('status', '')} |\n")

summary_json_path = os.path.join(OUTPUT_DIR, "summary.json")
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

zip_path = "/kaggle/working/all_results"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
print(f"Đã nén toàn bộ kết quả: {zip_path}.zip")
print(f"Đã tạo mục lục README.md và summary.json bên trong file zip!")


## Ghi chú

- **Về hiện tượng tên file tiếng Việt bị mất dấu trên Kaggle**:
  + **Nguyên nhân:** Khi tạo Dataset trên Kaggle và kéo thả trực tiếp file PDF có dấu tiếng Việt, trình duyệt và backend của Kaggle sẽ tự động thanh lọc bỏ các nguyên âm có dấu (`ô, ì, ú, ý, ữ, ả, đ...`), biến `Mô hình chú ý...` thành `M hnh ch  ng cnh...`.
  + **Cách tải dataset tối ưu nhất:** Nén toàn bộ các file PDF vào 1 file `.zip` (ví dụ `papers.zip`), sau đó upload file zip này lên Kaggle. Notebook đã được thiết kế sẵn tính năng **tự động phát hiện và giải nén file zip**, bảo đảm giữ nguyên vẹn 100% tên file tiếng Việt có dấu.
  + **Cơ chế tự động khôi phục:** Nếu file đã bị Kaggle làm biến dạng tên, notebook sử dụng PyMuPDF quét trang 1 để trích xuất dòng chữ có font-size lớn nhất (chính là Tiêu đề bài báo khoa học). Toàn bộ tên thư mục xuất ra (`00_Mo_hinh_chu_y_ngu_canh...`), file metadata `source_meta.json`, và tiêu đề `# Tiêu đề` trong Markdown đều được tự động khôi phục hoàn chỉnh.

- **Vấn đề công thức toán / bảng vẫn sai dù đã có VLM**: nguyên nhân là bộ lọc phát
  hiện "trang khó" ban đầu chỉ dựa trên độ dài text (trang ít ký tự = trang scan).
  Nhưng trang chứa công thức toán bị lỗi Unicode hay bảng dài vẫn có RẤT NHIỀU ký tự
  (chỉ là ký tự sai), nên bị bỏ sót, không được đưa qua VLM. Bước 5 đã sửa bằng cách
  dò thêm ký tự thuộc khối Unicode "Mathematical Alphanumeric Symbols" và heuristic
  nhận diện bảng — đây là 2 dấu hiệu cụ thể của đúng loại lỗi này.
- **Muốn chắc chắn 100% không bỏ sót trang nào**: đặt `FORCE_ALL_PAGES = True` ở
  Bước 5 để ép toàn bộ trang đi qua VLM, đổi lại thời gian chạy lâu hơn nhiều.
- **Chạy lại sau khi sửa bộ lọc**: không cần xóa gì cả — chỉ cần chạy lại Bước 5, 6, 7.
  `process_single_pdf` sẽ tự tính lại danh sách "trang khó" theo bộ lọc mới, và
  `vlm_progress.json` chỉ chứa các trang đã thực sự chạy VLM trước đó (nếu có) nên sẽ
  không bị bỏ sót trang mới phát hiện.
- **Nếu muốn xem trước bộ lọc đang phát hiện gì mà chưa chạy VLM**: gọi thử
  `classify_pages(pdf_files[INDEX])` và in ra để kiểm tra trước khi chạy toàn bộ Bước 7.
- **Lỗi `ImportError: cannot import name '_Ink' from 'PIL._typing'` (hoặc các lỗi
  ImportError lạ khác sau khi pip install)**: đây là hậu quả của việc nâng cấp thư
  viện giữa chừng session mà chưa restart kernel. Luôn chạy Bước 1.1 (restart kernel)
  ngay sau Bước 1, trước khi chạy bất kỳ ô nào khác. Nếu vẫn gặp lỗi sau khi đã
  restart, thử vào menu **Run > Restart & Clear Cell Outputs** của Kaggle rồi chạy
  lại từ đầu.
- **Lỗi "you are using a model of type qwen2_5_vl to instantiate qwen2_vl"**: đây là
  dấu hiệu dùng sai class model, đã được sửa ở Bước 3 bằng cách dùng đúng
  `Qwen2_5_VLForConditionalGeneration`. Nếu bạn copy code cũ từ nơi khác, luôn kiểm
  tra lại đúng class này.
- **Nếu vẫn OOM dù đã dùng đúng class**: gần như luôn do ảnh trang tạo ra quá nhiều
  token thị giác (không phải do bản thân model 4-bit) — hạ `MAX_PIXELS` ở Bước 3
  xuống nữa trước tiên (ví dụ `512*28*28`). Nếu vẫn chưa đủ, giảm tiếp `max_memory`
  xuống thấp hơn (ví dụ `"9GiB"` mỗi GPU) để chừa nhiều buffer hơn cho hoạt động sinh
  văn bản (generate), hoặc giảm `max_new_tokens` trong `ocr_page_with_vlm` (Bước 4)
  xuống 1024.
- **Nếu model bị chia lên nhiều GPU (nhìn `model.hf_device_map`)**: điều này bình
  thường và tự động được `accelerate` xử lý khi gọi `model.generate()`, không cần
  chỉnh gì thêm trong hàm `ocr_page_with_vlm`.
- **Về lỗi `MissingDependencyException` của markitdown (nếu vẫn gặp)**: lỗi này KHÔNG
  liên quan đến tên file tiếng Việt — nó xảy ra khi `markitdown` chưa được cài kèm
  extra `[pdf]`. Bước 1 ở trên đã cài đúng `pip install -U 'markitdown[pdf]'`, nên nếu
  bạn vẫn thấy lỗi này, hãy restart kernel Kaggle và chạy lại từ Bước 1 (có thể một
  bản `markitdown` cũ chưa có extra đã được cache trong session).
  tên file trong dataset, không phải lỗi của notebook. Nội dung PDF hoàn toàn không bị
  ảnh hưởng. Notebook cũng tự copy mỗi PDF sang tên `source.pdf` an toàn trước khi xử
  lý, để tránh mọi vấn đề tiềm ẩn khác với ký tự đặc biệt trong tên file gốc.
- **Nếu VRAM dư dả (≥ 20GB trống trên 1 GPU duy nhất)**: bỏ `quantization_config=bnb_config`
  ở Bước 3, dùng thẳng `torch_dtype=torch.bfloat16` — tốc độ và chất lượng OCR sẽ tốt hơn.
- **Nếu dataset có quá nhiều file / trang khó**: cân nhắc đặt `MAX_PAGES_PER_FILE`
  ở Bước 5 để tránh hết giờ session Kaggle (mỗi trang mất ~5-20 giây tùy độ phức tạp).
- **Chạy lại an toàn**: pipeline được thiết kế idempotent — chạy lại Bước 7 nhiều lần
  sẽ không xử lý lại các trang/file đã xong, nhờ `vlm_progress.json` mỗi thư mục con.
- **Nếu công thức toán vẫn ra sai**: thử tăng `DPI` lên 400-600 ở Bước 5 — an toàn về
  VRAM vì `MAX_PIXELS` ở Bước 3 giới hạn số token thị giác độc lập với DPI đầu vào
  (ảnh chỉ được render nét hơn trước khi bị co lại theo `MAX_PIXELS`).
- **Muốn xử lý thủ công 1 file cụ thể**: gọi trực tiếp
  `process_single_pdf(pdf_files[INDEX], "/kaggle/working/output/thu_muc_rieng")`
  thay vì chạy vòng lặp Bước 7.


- **Nếu chất lượng định dạng lại chưa đủ tốt so với ví dụ mong muốn**: model 7B chạy trên
  Kaggle có giới hạn về khả năng biên tập tinh tế (ví dụ: tự phát hiện công thức tự tham
  chiếu, sắp xếp lại đúng thứ tự mục phức tạp). Có thể cân nhắc lấy `final_combined.md` ra
  khỏi Kaggle và chạy bước định dạng lại bằng một model mạnh hơn (qua API hoặc chat) nếu
  cần độ chính xác biên tập cao hơn.
